# IOAI — 2026 Home Task Night Watch (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os
os.makedirs('data', exist_ok=True)
BASE='https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2026-home-task-night-watch'
for f in ['train.csv','test.csv']:
    if not os.path.exists(f'data/{f}'): os.system(f'wget -q {BASE}/{f} -O data/{f}')
# 오디오(1.5GB)·AST 모델은 노트북 첫 셀이 Drive 에서 직접 받는다.
print('CSV 준비:', [f for f in ['train.csv','test.csv'] if os.path.exists('data/'+f)])
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Operation Night Watch — IOAI-2026 Home Task 1

**지속학습(continual learning)** 문제. 16개 기존 소리(old)로 사전학습된 **AST**(Audio Spectrogram Transformer)가
주어진다. 여기에 13개 신규 소리(new)를 추가로 인식하되 **기존을 잊지 않아야** 한다. 평가지표는
**½·Acc_old + ½·Acc_new** (신규만 배우고 기존을 잊으면 0.5 이하).

`data/train.csv`(29클래스 라벨)로 학습하고 `data/test.csv` 를 예측해 `submission.csv`(`path,target`)를 만든다.
사전학습 모델·오디오는 위 셀이 Drive 에서 자동 다운로드한다(GPU 필요).

In [ ]:
# 오디오(~1.5GB)와 사전학습 AST 모델은 공개 Google Drive 에서 받아 data/ 에 푼다(최초 1회).
!pip install -q gdown soundfile
import os, zipfile, random, gdown
os.makedirs("data", exist_ok=True)
if not os.path.isdir("data/audio"):
    MIRRORS = ["12XaRWYDsHq3eBwrgZ0QnEOffflS_dlff",
               "1iUYDtKxXM1GSG5VNJ3r9847hqhWEiQRq",
               "112F9IFfn3_UtOPwSM1cGbuAL7Ci2aBP_"]
    random.shuffle(MIRRORS)
    for fid in MIRRORS:
        try:
            gdown.download(id=fid, output="dataset.zip", quiet=False); break
        except Exception as e:
            print("mirror failed, trying next:", e)
    with zipfile.ZipFile("dataset.zip") as z:
        for n in z.namelist():
            if (n.startswith("audio/") or n.startswith("model/")) and not n.startswith("__MACOSX"):
                z.extract(n, "data")
print("audio files:", len(os.listdir("data/audio")), "| model:", sorted(os.listdir("data/model")))

In [ ]:
import pandas as pd, numpy as np, torch, soundfile as sf
from transformers import ASTForAudioClassification, ASTFeatureExtractor
dev = "cuda" if torch.cuda.is_available() else "cpu"
SR = 16000
fe = ASTFeatureExtractor.from_pretrained("data/model")
ast = ASTForAudioClassification.from_pretrained("data/model").to(dev).eval()   # 16 old classes 로 사전학습

def _load(path):
    w, sr = sf.read(os.path.join("data", path))
    if w.ndim > 1: w = w.mean(1)
    w = w.astype(np.float32)
    if sr != SR:
        n = int(len(w) * SR / sr); w = np.interp(np.linspace(0, len(w), n, endpoint=False), np.arange(len(w)), w).astype(np.float32)
    return w

@torch.no_grad()
def embed_and_logits(paths, bs=16):
    E, L = [], []
    for i in range(0, len(paths), bs):
        wavs = [_load(p) for p in paths[i:i+bs]]
        inp = fe(wavs, sampling_rate=SR, return_tensors="pt").to(dev)
        o = ast(**inp, output_hidden_states=True)
        E.append(o.hidden_states[-1].mean(1).cpu().numpy()); L.append(o.logits.cpu().numpy())
    return np.concatenate(E), np.concatenate(L)

train = pd.read_csv("data/train.csv")   # path, target(0-28), category
test  = pd.read_csv("data/test.csv")    # path
print("train", len(train), "| test", len(test))

In [ ]:
# 베이스라인: 사전학습 AST 를 그대로 사용(16 old 만 예측 가능 → 신규는 전부 틀림 → 점수 ≈ 0.5)
_, logits = embed_and_logits(test.path.tolist())
pred = logits.argmax(1)                        # 0..15 only
pd.DataFrame({"path": test["path"], "target": pred.astype(int)}).to_csv("submission.csv", index=False)
print("saved submission.csv", len(pred), "(baseline: old only)")

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)